In [ ]:
# Qdrant
from qdrant_client import QdrantClient
print(QdrantClient(url="http://localhost:6333").get_collections())

In [ ]:
"""
### Ollama / Llama 3.1
from langchain_ollama import ChatOllama
print(ChatOllama(model="llama3.1").invoke("ok?"))
"""

"""
### Ollama / Llama 3.1 (con streaming)
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1")

# Stampa i token man mano che vengono generati
for chunk in llm.stream("ok?"):
    print(chunk.content, end="", flush=True)
print()
"""

### Ollama / Llama 3.2: (con streaming)
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    keep_alive="1h",
    num_thread=4 #fondamentale per evitare bottleneck
)

for chunk in llm.stream("Rispondi con una frase: cos'è un grafo?"):
    print(chunk.content, end="", flush=True)

In [ ]:
# NetworkX
import networkx as nx
G = nx.Graph(); G.add_edge("a", "b", weight=0.9)
print(nx.node_link_data(G))

In [ ]:
# anywidget
import anywidget, traitlets
from IPython.display import display

class _T(anywidget.AnyWidget):
    _esm = "function render({el}){ el.innerHTML = '✅ anywidget ok'; } export default {render};"
_T()
display(_T())

In [ ]:
# anywidget > grafo di prova
import anywidget
import traitlets
import networkx as nx

# 1. Creazione del grafo di test NetworkX
G = nx.Graph()
G.add_node("chunk_a", text="GraphRAG e Vector DB")
G.add_node("chunk_b", text="Qdrant per la ricerca")
G.add_node("chunk_c", text="NetworkX per la struttura")
G.add_edge("chunk_a", "chunk_b", weight=0.9)
G.add_edge("chunk_a", "chunk_c", weight=0.75)

# Convertiamo il grafo NetworkX in JSON compatibile
graph_json = nx.node_link_data(G)
# Se nx.node_link_data genera la chiave 'edges', la rinominiamo in 'links' per D3.js
if "edges" in graph_json:
    graph_json["links"] = graph_json.pop("edges")

# 2. Definizione della Classe AnyWidget con D3.js integrato
class GraphWidget(anywidget.AnyWidget):
    _esm = """
    import * as d3 from "https://esm.sh/d3@7";

    export function render({ model, el }) {
      el.innerHTML = "";
      
      const width = 600;
      const height = 350;
      
      const svg = d3.select(el).append("svg")
        .attr("width", width)
        .attr("height", height)
        .style("background", "#f8fafc")
        .style("border", "1px solid #cbd5e1")
        .style("border-radius", "8px");

      function draw() {
        const graph = model.get("graph_data");
        if (!graph || !graph.nodes || graph.nodes.length === 0) return;

        svg.selectAll("*").remove();

        const nodes = graph.nodes.map(d => ({ ...d }));
        const links = graph.links ? graph.links.map(d => ({ ...d })) : [];

        const simulation = d3.forceSimulation(nodes)
          .force("link", d3.forceLink(links).id(d => d.id).distance(100))
          .force("charge", d3.forceManyBody().strength(-180))
          .force("center", d3.forceCenter(width / 2, height / 2));

        const link = svg.append("g")
          .selectAll("line")
          .data(links)
          .enter().append("line")
          .attr("stroke", "#94a3b8")
          .attr("stroke-width", 2);

        const node = svg.append("g")
          .selectAll("circle")
          .data(nodes)
          .enter().append("circle")
          .attr("r", 12)
          .attr("fill", "#6366f1")
          .style("cursor", "pointer")
          .on("click", (event, d) => {
             model.set("selected_node", { id: d.id, text: d.text || "" });
             model.save_changes();
          });

        const label = svg.append("g")
          .selectAll("text")
          .data(nodes)
          .enter().append("text")
          .text(d => d.id)
          .attr("font-size", "11px")
          .attr("dx", 15)
          .attr("dy", 4);

        simulation.on("tick", () => {
          link
            .attr("x1", d => d.source.x)
            .attr("y1", d => d.source.y)
            .attr("x2", d => d.target.x)
            .attr("y2", d => d.target.y);

          node
            .attr("cx", d => d.x)
            .attr("cy", d => d.y);

          label
            .attr("x", d => d.x)
            .attr("y", d => d.y);
        });
      }

      model.on("change:graph_data", draw);
      draw();
    }
    """
    
    # Proprietà reattive sincronizzate
    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_node = traitlets.Dict({}).tag(sync=True)

# 3. Istanziazione del widget e caricamento dati
widget = GraphWidget()
widget.graph_data = graph_json
widget

In [ ]:
# anywidget -- classi collegate
import sys
from pathlib import Path

# Aggiunge la cartella radice del progetto al sys.path per importare da src/
sys.path.append(str(Path.cwd()))

from src.graph_builder import KnowledgeGraphBuilder
from src.chunk_widget import ChunkGraphWidget

# 1. Costruzione del grafo tramite la classe modulare
builder = KnowledgeGraphBuilder()
builder.add_chunk_node("chunk_0", text="GraphRAG combina Vector DB e Grafi di Conoscenza.")
builder.add_chunk_node("chunk_1", text="Qdrant gestisce la ricerca vettoriale ad alte prestazioni.")
builder.add_chunk_node("chunk_2", text="NetworkX modella la struttura del grafo nel backend Python.")

builder.add_relation("chunk_0", "chunk_1", weight=0.85)
builder.add_relation("chunk_0", "chunk_2", weight=0.78)

# 2. Istanziazione e visualizzazione del widget
widget = ChunkGraphWidget()
widget.graph_data = builder.to_json_data()
widget

In [3]:
### CHECKS AND STARTS ALL ARCHITECTURE
import sys
import time
import subprocess
from pathlib import Path

# Assicura l'importazione dei moduli dalla cartella src/
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

import qdrant_client
from qdrant_client.models import VectorParams, Distance, PointStruct

# Importazione client LLM Ollama e FastEmbed per l'embedding in memoria Python
from langchain_ollama import ChatOllama
from langchain_community.embeddings import FastEmbedEmbeddings

from src.graph_builder import KnowledgeGraphBuilder
from src.chunk_widget import ChunkGraphWidget

# ----------------------------------------------------------------------
# IMPORT CONFIGURAZIONE GLOBALE DA src/config.py
# ----------------------------------------------------------------------
from src.config import (
    LLM_MODEL,
    EMBEDDING_MODEL,
    QDRANT_URL,
    DEFAULT_KEEP_ALIVE,
    DEFAULT_NUM_THREAD
)

# Avvio timer totale della pipeline
total_start_time = time.perf_counter()

print(f"^^^ Configurazione Attiva -> LLM: '{LLM_MODEL}' (Ollama) | Embeddings: '{EMBEDDING_MODEL}' (FastEmbed)")

print("\n=== 1. VERIFICA E AVVIO SERVIZI BACKEND ===")
def check_or_start_service(process_name, command):
    check = subprocess.run(["pgrep", "-f", process_name], capture_output=True)
    if not check.stdout:
        print(f">> Avvio di {process_name} in corso...")
        subprocess.Popen(command, shell=True)
        time.sleep(2)
    else:
        print(f"!>> {process_name} è già attivo.")

check_or_start_service("qdrant", "cd ~/tesi_graphrag/data && nohup ~/.local/bin/qdrant > ~/tesi_graphrag/logs/qdrant.log 2>&1 &")
check_or_start_service("ollama serve", "nohup ~/.local/bin/ollama serve > ~/tesi_graphrag/logs/ollama.log 2>&1 &")

# Inizializzazione Client Qdrant, Embedding FastEmbed e LLM Ollama
client = qdrant_client.QdrantClient(url=QDRANT_URL)
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)
llm = ChatOllama(
    model=LLM_MODEL, 
    temperature=0, 
    keep_alive=DEFAULT_KEEP_ALIVE, 
    num_thread=DEFAULT_NUM_THREAD
)

print("\n=== 2. VECTOR STORE: INDICIZZAZIONE E SEARCH ===")
collection_name = "test_rag_chunks"

# Calcolo veloce della dimensione del vettore in locale (ONNX Runtime)
vector_dim = len(embeddings.embed_query("test"))
print(f"!>> Dimensione embedding rilevata ({EMBEDDING_MODEL}): {vector_dim}D")

# Reset/Creazione collezione Qdrant
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE)
)

# Chunk di test
test_chunks = [
    {"id": 0, "text": "GraphRAG combina l'uso di Vector DB e Grafi di Conoscenza per migliorare il recupero delle informazioni."},
    {"id": 1, "text": "Qdrant è un database vettoriale ad alte prestazioni ottimizzato per ricerche di similarità."},
    {"id": 2, "text": "NetworkX consente di rappresentare la struttura relazionale dei chunk sotto forma di grafo in Python."}
]

# 1. VETTORIZZAZIONE BATCH IN MEMORIA PYTHON (FastEmbed)
embed_start = time.perf_counter()
texts_to_embed = [item["text"] for item in test_chunks]
vectors = embeddings.embed_documents(texts_to_embed) # Chiamata BATCH locale
embed_time = time.perf_counter() - embed_start

# Inserimento punti in Qdrant
points = [
    PointStruct(id=item["id"], vector=vec, payload=item)
    for item, vec in zip(test_chunks, vectors)
]

client.upsert(collection_name=collection_name, points=points)
print(f"!>> Inseriti {len(points)} chunk vettorizzati su Qdrant in BATCH.")
print(f"<<< Tempo Vettorizzazione Batch: {embed_time:.3f} s")

# 2. QUERY SEMANTICA LOCALE + RETRIEVAL (Zero delay)
query = "Come si collegano i grafi ai database vettoriali?"
retrieval_start = time.perf_counter()

query_vector = embeddings.embed_query(query)
search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=2
).points

retrieved_texts = [hit.payload["text"] for hit in search_results]
retrieval_time = time.perf_counter() - retrieval_start

print(f"?> Query: '{query}'")
print(f"!>> Chunk recuperati da Qdrant: {len(retrieved_texts)}")
print(f"<<< Tempo Retrieval Qdrant: {retrieval_time:.3f} s")

print("\n=== 3. GENERAZIONE RISPOSTA LLM (RAG) ===")
prompt = f"Rispondi brevemente alla seguente domanda basandoti solo sul contesto fornito:\nDomanda: {query}\nContesto:\n" + "\n".join(retrieved_texts)

# 3. TIMER GENERAZIONE LLM (Ollama gestisce solo Llama 3.2 in RAM)
llm_start = time.perf_counter()
response = llm.invoke(prompt)
llm_time = time.perf_counter() - llm_start

print(f"???> Risposta ({LLM_MODEL}):\n{response.content.strip()}")
print(f"<<< Tempo Generazione LLM: {llm_time:.3f} s")

print("\n=== 4. KNOWLEDGE GRAPH BUILDER & ANYWIDGET ===")
graph_start = time.perf_counter()

# Inizializzazione della classe backend per il grafo
builder = KnowledgeGraphBuilder()

# Inseriamo i nodi ed un arco di similarità basato sul recupero
for item in test_chunks:
    builder.add_chunk_node(chunk_id=f"chunk_{item['id']}", text=item["text"])

# Collegamento tra il chunk 0 e gli altri due
builder.add_relation("chunk_0", "chunk_1", weight=0.85)
builder.add_relation("chunk_0", "chunk_2", weight=0.78)

# Conversione e instanziazione Widget
widget = ChunkGraphWidget()
widget.graph_data = builder.to_json_data()
graph_time = time.perf_counter() - graph_start

total_time = time.perf_counter() - total_start_time

# ----------------------------------------------------------------------
# RIEPILOGO METRICHE DI PRESTAZIONE
# ----------------------------------------------------------------------
print("\n" + "="*45)
print("<<< RIEPILOGO PRESTAZIONI PIPELINE")
print("="*45)
print(f"  • Vettorizzazione Batch: {embed_time:.3f} s")
print(f"  • Retrieval Vector DB:  {retrieval_time:.3f} s")
print(f"  • Generazione LLM:      {llm_time:.3f} s")
print(f"  • Costruzione Grafo:    {graph_time:.3f} s")
print("-" * 45)
print(f"⏱️ TEMPO TOTALE:          {total_time:.3f} s")
print("="*45)

print("\n!>> Test completato con successo. Visualizzazione del widget D3.js qui sotto:")
widget

^^^ Configurazione Attiva -> LLM: 'llama3.2' (Ollama) | Embeddings: 'nomic-ai/nomic-embed-text-v1.5' (FastEmbed)

=== 1. VERIFICA E AVVIO SERVIZI BACKEND ===
!>> qdrant è già attivo.
!>> ollama serve è già attivo.

=== 2. VECTOR STORE: INDICIZZAZIONE E SEARCH ===
!>> Dimensione embedding rilevata (nomic-ai/nomic-embed-text-v1.5): 768D
!>> Inseriti 3 chunk vettorizzati su Qdrant in BATCH.
<<< Tempo Vettorizzazione Batch: 0.780 s
?> Query: 'Come si collegano i grafi ai database vettoriali?'
!>> Chunk recuperati da Qdrant: 2
<<< Tempo Retrieval Qdrant: 0.496 s

=== 3. GENERAZIONE RISPOSTA LLM (RAG) ===
???> Risposta (llama3.2):
I grafi vengono collegati ai database vettoriali utilizzando l'interfaccia di GraphRAG, che fornisce una connessione tra i grafi di conoscenza e i database vettoriali come Qdrant.
<<< Tempo Generazione LLM: 7.698 s

=== 4. KNOWLEDGE GRAPH BUILDER & ANYWIDGET ===

<<< RIEPILOGO PRESTAZIONI PIPELINE
  • Vettorizzazione Batch: 0.780 s
  • Retrieval Vector DB:  0.496 s